In [ ]:
import numpy as np

# ---------- 1. Data preparation ----------
def train_test_split_np(X, y, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))
    cut = int(len(y) * (1 - test_size))
    return X[idx[:cut]], X[idx[cut:]], y[idx[:cut]], y[idx[cut:]]

def standardize(X_tr, X_te):
    """z = (x - mean) / std, with mean and std computed on the training part only."""
    mu, sigma = X_tr.mean(axis=0), X_tr.std(axis=0)
    sigma[sigma == 0] = 1.0
    return (X_tr - mu) / sigma, (X_te - mu) / sigma

# ---------- 2. Model: logistic regression ----------
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def train_logreg(X, y, lr=0.1, n_iter=200):
    n, d = X.shape
    w, b = np.zeros(d), 0.0
    for _ in range(n_iter):
        p = sigmoid(X @ w + b)
        w -= lr * (X.T @ (p - y)) / n
        b -= lr * np.mean(p - y)
    return w, b

def predict(X, w, b):
    return (sigmoid(X @ w + b) >= 0.5).astype(int)

# ---------- 3. Metric ----------
def accuracy_score(y_true, y_pred):
    return np.mean(y_true == y_pred)

# ---------- 4. Cross-validated accuracy of a subset ----------
def kfold_indices(n, K=5, seed=42):
    rng = np.random.default_rng(seed)
    return np.array_split(rng.permutation(n), K)

def accuracy(mask, X, y, K=5, seed=42):
    """Output: one number in [0, 1], the CV accuracy using only features where mask == 1."""
    mask = np.asarray(mask, dtype=bool)
    if not mask.any():                       # no feature selected
        return 0.0
    Xs = X[:, mask]
    folds = kfold_indices(len(y), K, seed)
    scores = []
    for k in range(K):
        te = folds[k]
        tr = np.concatenate([folds[j] for j in range(K) if j != k])
        Xtr, Xte = standardize(Xs[tr], Xs[te])
        w, b = train_logreg(Xtr, y[tr])
        scores.append(accuracy_score(y[te], predict(Xte, w, b)))
    return np.mean(scores)

# ---------- 5. Fitness used by TS, GA and SA ----------
class Fitness:
    def __init__(self, X, y, lam=0.01):
        self.X, self.y, self.lam = X, y, lam
        self.cache = {}
        self.n_evals = 0                     # real evaluations (for the equal-budget comparison)

    def __call__(self, mask):
        key = tuple(int(m) for m in mask)
        if key not in self.cache:
            self.n_evals += 1
            acc = accuracy(mask, self.X, self.y)
            self.cache[key] = acc - self.lam * (sum(key) / len(key))
        return self.cache[key]

In [ ]:
from sklearn.datasets import load_breast_cancer   # only to load the data

X, y = load_breast_cancer(return_X_y=True)
X, y = X.astype(float), y.astype(int)

X_train, X_test, y_train, y_test = train_test_split_np(X, y)

fit = Fitness(X_train, y_train, lam=0.01)

n = X_train.shape[1]
all_features = np.ones(n)
rng = np.random.default_rng(0)
random_subset = rng.integers(0, 2, n)

print(accuracy(all_features, X_train, y_train))    # baseline, should be roughly 0.95 to 0.98
print(accuracy(random_subset, X_train, y_train))
print(fit(random_subset))                          # accuracy minus the penalty

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt

# 1. Load Dataset (30 features)
data = load_breast_cancer()
X, y = data.data, data.target
n_features = X.shape[1]

# 2. Fitness Function
def calculate_fitness(solution, X, y, alpha=0.99):
    """
    Evaluates solution. Maximizes accuracy while slightly penalizing large feature sets.
    """
    selected_indices = np.where(solution == 1)[0]
    if len(selected_indices) == 0:
        return 0.0 # Invalid solution penalty
        
    X_subset = X[:, selected_indices]
    clf = KNeighborsClassifier(n_neighbors=5)
    accuracy = np.mean(cross_val_score(clf, X_subset, y, cv=5))
    
    # Fitness = alpha * accuracy + (1-alpha) * (1 - (selected_features / total_features))
    fitness = alpha * accuracy + (1 - alpha) * (1 - len(selected_indices) / n_features)
    return fitness

# 3. Neighborhood Generation (Swap)
def generate_swap_neighbors(solution):
    """Generates neighbors by swapping one '1' with one '0'."""
    neighbors = []
    moves = []
    ones = np.where(solution == 1)[0]
    zeros = np.where(solution == 0)[0]
    
    for o in ones:
        for z in zeros:
            neighbor = solution.copy()
            neighbor[o] = 0
            neighbor[z] = 1
            neighbors.append(neighbor)
            moves.append((o, z)) # Track the move (dropped feature, added feature)
    return neighbors, moves

# 4. Tabu Search Algorithm
def tabu_search(X, y, max_iter=50, tabu_tenure=5, n_initial_features=15):
    # Initialization
    current_solution = np.zeros(n_features, dtype=int)
    current_solution[np.random.choice(n_features, n_initial_features, replace=False)] = 1
    best_global_solution = current_solution.copy()
    best_global_fitness = calculate_fitness(best_global_solution, X, y)
    
    tabu_list = [] # Stores recent moves
    long_term_memory = np.zeros(n_features) # Tracks frequency of feature flips
    
    history_best = []
    
    for iteration in range(max_iter):
        neighbors, moves = generate_swap_neighbors(current_solution)
        
        best_neighbor = None
        best_neighbor_fitness = -1
        best_move = None
        
        # Evaluate Neighborhood
        for neighbor, move in zip(neighbors, moves):
            fit = calculate_fitness(neighbor, X, y)
            
            # Diversification: Penalize frequently flipped features in local evaluation
            penalty = 0.001 * (long_term_memory[move[0]] + long_term_memory[move[1]])
            adjusted_fit = fit - penalty
            
            is_tabu = move in tabu_list or (move[1], move[0]) in tabu_list
            
            # Aspiration Criterion: Accept if tabu but beats global best
            if is_tabu and fit > best_global_fitness:
                is_tabu = False 
                
            if not is_tabu and adjusted_fit > best_neighbor_fitness:
                best_neighbor = neighbor
                best_neighbor_fitness = adjusted_fit
                best_move = move
                unadjusted_fit = fit
                
        if best_neighbor is None:
            break # No valid non-tabu moves left
            
        # Move to best neighbor
        current_solution = best_neighbor
        
        # Update Long-Term Memory
        long_term_memory[best_move[0]] += 1
        long_term_memory[best_move[1]] += 1
        
        # Update Tabu List
        tabu_list.append(best_move)
        if len(tabu_list) > tabu_tenure:
            tabu_list.pop(0)
            
        # Update Global Best
        if unadjusted_fit > best_global_fitness:
            best_global_solution = current_solution.copy()
            best_global_fitness = unadjusted_fit
            
        history_best.append(best_global_fitness)
        print(f"Iteration {iteration+1}/{max_iter} | Best Fitness: {best_global_fitness:.4f} | Features: {sum(best_global_solution)}")
        
    return best_global_solution, history_best

# Run the algorithm
best_sol, history = tabu_search(X, y, max_iter=30, tabu_tenure=7)

# Visualization for the Notebook
plt.plot(history, marker='o')
plt.title("Tabu Search Convergence")
plt.xlabel("Iteration")
plt.ylabel("Best Fitness (Accuracy & Sparsity)")
plt.grid(True)
plt.show()